In [1]:
import numpy as np
import pandas as pd

from assignment_1.modules.models import seen_by_user_rule, unseen_item_rule
from assignment_1.modules.util import load_csv



In [2]:
DATA_DIR, DIST = "../data", "../dist"
TASKS = {
    2: ("task_2_test.csv", "tasks_1_2_train.csv"),
    4: ("task_4_test.csv", "tasks_3_4_train.csv"),
    5: ("task_5_test.csv", "task_5_train.csv"),
}
data = {n: (load_csv(te, DATA_DIR), load_csv(tr, DATA_DIR)) for n, (te, tr) in TASKS.items()}

In [3]:
rows = []
for n, (test, train) in data.items():
    u1, u2 = ~test["item1"].isin(train["item"]), ~test["item2"].isin(train["item"])
    hist = set(train["user"] + "|" + train["item"])
    s1 = (test["user"] + "|" + test["item1"]).isin(hist)
    s2 = (test["user"] + "|" + test["item2"]).isin(hist)
    rows.append({"task": n,
                 "unseen: one": (u1 ^ u2).mean(), "unseen: both": (u1 & u2).mean(),
                 "in history: one": (s1 ^ s2).mean(), "in history: both": (s1 & s2).mean()})
pd.DataFrame(rows).set_index("task").map("{:.2%}".format)

,unseen: one,unseen: both,in history: one,in history: both
task,,,,
2,0.00%,0.00%,1.92%,0.00%
4,0.00%,0.00%,5.07%,0.00%
5,3.96%,0.00%,0.69%,0.00%


In [4]:
rows = []
for n, (test, train) in data.items():
    pred = pd.read_csv(f"{DIST}/task_{n}_predictions.csv")["which"].to_numpy()
    for name, (applies, choice) in [("unseen", unseen_item_rule(train, test)),
                                    ("in history", seen_by_user_rule(train, test))]:
        wrong = (pred != choice)[applies].mean() if applies.any() else np.nan
        rows.append({"task": n, "rule": name, "coverage": applies.mean(),
                     "dist disagrees": wrong, "expected gain": applies.mean() * wrong})
pd.DataFrame(rows).set_index(["task", "rule"]).map(lambda x: f"{x:.2%}")

coverage dist disagrees expected gain
task rule                                            
2    unseen        0.00%           nan%          nan%
     in history    1.92%         54.05%         1.04%
4    unseen        0.00%           nan%          nan%
     in history    5.07%          0.00%         0.00%
5    unseen        3.96%          0.00%         0.00%
     in history    0.69%         61.87%         0.43%